# OpenMontage Kaggle Batch Asset Generator

This kernel runs on Kaggle's GPU infrastructure and generates all image/video
assets for a single OpenMontage project in one session. It is submitted by the
Codespace tool layer and polled until completion.

## Behavior contract

- Reads job tasks directly from the embedded `JOB_TASKS` dict (no external files).
- Detects GPU compute capability and selects the appropriate model branch.
- Hard-fails on any unrecoverable error — never silently substitutes placeholders.

In [ ]:
import os, sys, json, time, shutil, base64, hashlib
from pathlib import Path
from datetime import datetime, timezone
import torch

# ============================================================================
# HF_TOKEN FALLBACK CHAIN
# Priority: Kaggle Secrets > os.environ > empty string
# Kaggle kernels do NOT inherit the Codespace .env, so the user MUST set
# HF_TOKEN as an actual Kaggle Secret (Settings -> Add-ons -> Secrets).
# ============================================================================
HF_TOKEN = os.environ.get("HF_TOKEN", "")

# Try Kaggle secrets next - this is the primary expected path for Kaggle kernels
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    hf_from_secrets = user_secrets.get_secret("HF_TOKEN")
    if hf_from_secrets:
        HF_TOKEN = hf_from_secrets
except Exception:
    pass

if not HF_TOKEN:
    print("WARNING: HF_TOKEN is empty. HF Hub downloads may hit rate limits.")

# Set token for huggingface_hub
os.environ["HF_TOKEN"] = HF_TOKEN

In [ ]:
# ============================================================================
# GPU DETECTION
# Determines which model branch to use based on compute capability.
print("Detecting GPU...")
GPU_BRANCH = "cpu"
GPU_NAME = "cpu"
DEVICE = "cpu"

if torch.cuda.is_available():
    cc = torch.cuda.get_device_properties(0).major
    GPU_NAME = torch.cuda.get_device_name(0)
    
    if cc >= 7:
        GPU_BRANCH = "t4_or_better"
        DEVICE = "cuda"
        print(f"GPU detected: {GPU_NAME} (compute capability {cc})")
        print("Branch: SANA-Sprint (steps=4, guidance_scale=4.5, 1024x576)")
    elif cc == 6:
        GPU_BRANCH = "p100"
        DEVICE = "cuda"
        print(f"GPU detected: {GPU_NAME} (compute capability {cc})")
        print("Branch: FLUX.1 Schnell FP8 (P100 incompatible with SANA)")
    else:
        GPU_BRANCH = "old_gpu"
        DEVICE = "cpu"
        print(f"GPU detected but CC={cc} not supported, falling back to CPU")
else:
    print("No GPU detected. Branch: FLUX.1 Schnell FP8 CPU-only mode")

print(f"Active branch: {GPU_BRANCH}")
print(f"Device: {DEVICE}")
print(f"HF_TOKEN present: {bool(HF_TOKEN)}")

In [ ]:
# ============================================================================# SINGLETON MODEL LOADING# Load pipelines once and reuse across all assets in the batch_SANA_PIPE = None_FLUX_PIPE = Nonedef _get_sana_pipe():    """Load SANA-Sprint pipeline once per kernel session (singleton pattern)."""    global _SANA_PIPE    if _SANA_PIPE is None:        from diffusers import SanaSprintPipeline        import torch        _SANA_PIPE = SanaSprintPipeline.from_pretrained(            "Efficient-Large-Model/Sana_Sprint_1.6B_1024px_diffusers",            torch_dtype=torch.bfloat16 if DEVICE == "cuda" else torch.float32,        )    return _SANA_PIPEdef _get_flux_pipe():    """Load FLUX pipeline once per kernel session (singleton pattern)."""    global _FLUX_PIPE    if _FLUX_PIPE is None:        from diffusers import FluxPipeline        import torch        _FLUX_PIPE = FluxPipeline.from_pretrained(            "black-forest-labs/FLUX.1-schnell",            torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,            device_map='cuda' if DEVICE == "cuda" else None,        )        if GPU_BRANCH == "p100":            _FLUX_PIPE.enable_quantization()        if DEVICE == "cpu":            _FLUX_PIPE.enable_model_cpu_offload()        # Warm-up and offload text encoder to CPU to save ~1.5GB VRAM        with torch.no_grad():            _ = _FLUX_PIPE.text_encoder.encode("test")            torch.cuda.empty_cache()            import gc            gc.collect()    return _FLUX_PIPE

In [ ]:
# ============================================================================
# EMBEDDED JOB TASKS
# Each project's job tasks are injected directly into this notebook by the
# Codespace submitter before kernel push. This avoids the unreliable
# /kaggle/working/ file system for job configuration.
# ============================================================================

# Example structure of JOB_TASKS (set by Codespace submitter):
# {
#   "project_id": "dancing-plague-1518",
#   "channel": "dark-annals",
#   "assets": [
#       {"type": "image", "id": "scene_001", "prompt": "...", "width": 1024, "height": 576},
#       ...
#   ]
# }

# PLACEHOLDER - will be overwritten by the Codespace submitter tool
JOB_TASKS = {
    "project_id": "PLACEHOLDER",
    "channel": "dark-annals",
    "assets": []
}

OUTPUT_DIR = Path("/kaggle/working/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Job tasks loaded: {len(JOB_TASKS.get('assets', []))} assets to generate")

## Hard-fail safeguard

Any generation failure stops the job immediately and returns a non-zero exit
code. There is NO silent substitution of placeholder images. The caller should
inspect the generated manifest and abort on any missing asset.

In [ ]:
def write_manifest(results: list, error: str | None = None) -> dict:
    """Write the asset manifest to disk.
    
    Returns the manifest dict. If error is set, the manifest includes
    failure markers for every asset.
    """
    manifest = {
        "project_id": JOB_TASKS["project_id"],
        "channel": JOB_TASKS["channel"],
        "gpu_branch": GPU_BRANCH,
        "gpu_name": GPU_NAME,
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "error": error,
        "assets": results,
    }
    manifest_path = OUTPUT_DIR / "manifest.json"
    manifest_path.write_text(json.dumps(manifest, indent=2))
    return manifest


def fail_job(message: str) -> None:
    """Hard-fail the job with a clear error message."""
    print(f"[FATAL] {message}")
    write_manifest([], error=message)
    sys.exit(1)

In [ ]:
# ============================================================================
# IMAGE GENERATION
# Uses SANA-Sprint on T4+, FLUX.1 Schnell FP8 on P100 or CPU.
# ============================================================================

from PIL import Image
import io

def channel_to_negative_prompt(channel: str) -> str:
    """Map channel styles to anti-anachronism negative prompts."""
    negatives = {
        "dark-annals": "modern clothing, electric lighting, smartphones, digital screens, anachronistic architecture, contemporary technology",
        "crime-ledger": "contemporary fashion, neon signs, mixed-era tech, photorealistic",
        "mind-tactics": "futuristic interfaces, holograms, cyberpunk accessories, neon",
    }
    return negatives.get(channel, "")

def generate_image(task: dict) -> dict:
    """Generate a single image asset with VRAM/duration tracking."""
    import time
    asset_id = task["id"]
    channel = JOB_TASKS.get("channel", "unknown")
    negative_prompt = channel_to_negative_prompt(channel)
    
    # Load pipeline once per session (singleton)
    pipe = _get_sana_pipe() if GPU_BRANCH == "t4_or_better" else _get_flux_pipe()
    
    # Start timer AFTER pipe retrieval (excludes one-time model load)
    start = time.time()
    
    # Reset memory stats before each generation
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
    
    with torch.no_grad():
        if GPU_BRANCH == "t4_or_better":
            image = pipe(
                prompt=task["prompt"],
                height=task.get("height", 576),
                width=task.get("width", 1024),
                negative_prompt=negative_prompt,
                num_inference_steps=4,
                guidance_scale=4.5,
            ).images[0]
        else:
            image = pipe(
                prompt=task["prompt"],
                height=task.get("height", 576),
                width=task.get("width", 1024),
                negative_prompt=negative_prompt,
                num_inference_steps=4,
                guidance_scale=0.0,
            ).images[0]
    
    # Calculate metrics
    duration_s = time.time() - start
    mem_used = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else 0
    
    # Cleanup
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    import gc
    gc.collect()
    
    out_path = OUTPUT_DIR / f"{asset_id}.png"
    image.save(out_path)
    
    print(f"  VRAM peak: {mem_used/1e9:.2f} GB, duration: {duration_s:.2f}s")
    
    return {
        "id": asset_id,
        "type": "image",
        "path": str(out_path),
        "success": True,
        "memory_peak_bytes": mem_used,
        "duration_s": round(duration_s, 2),
    }


In [ ]:
# ============================================================================
# MAIN GENERATION LOOP
# ============================================================================

def main():
    if not JOB_TASKS.get("assets"):
        print("No assets to generate.")
        write_manifest([], error="empty_job")
        sys.exit(0)
    
    results = []
    assets = JOB_TASKS["assets"]
    total = len(assets)
    
    for idx, task in enumerate(assets, 1):
        asset_id = task.get("id", f"asset_{idx}")
        asset_type = task.get("type", "image")
        print(f"[{idx}/{total}] Generating {asset_type}: {asset_id}")
        
        try:
            if asset_type == "image":
                result = generate_image(task)
            else:
                fail_job(f"Unsupported asset type: {asset_type}")
            
            if not result.get("success"):
                fail_job(f"Generation failed for {asset_id}: {result}")
            
            # Verify output file actually exists
            out = Path(result["path"])
            if not out.exists() or out.stat().st_size < 1024:
                fail_job(f"Generated file missing or empty for {asset_id}: {out}")
            
            results.append(result)
            print(f"  OK -> {out} ({out.stat().st_size} bytes)")
            
        except Exception as exc:
            fail_job(f"Exception generating {asset_id}: {exc}")
    
    manifest = write_manifest(results)
    print(f"\nComplete. Generated {len(results)}/{total} assets.")
    print(f"Manifest: {OUTPUT_DIR / 'manifest.json'}")


if __name__ == "__main__":
    main()